# Build a Knowledge Graph from Text

Extract **people, companies, positions, skills, hobbies, and relationships** from a company profile document automatically.

In [ ]:
!pip install langchain-core langchain-openai neo4j python-dotenv chromadb pyvis pydantic

# Start Jupyter Lab

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Load API key from any .env location
for p in [Path("."), Path(".."), Path("../.."), Path("../../..")]:
    load_dotenv(p.resolve() / ".env")

# Neo4j connection
NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "workshop2024")

# Connect and clear all existing data
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
    count = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
driver.close()

print(f"Neo4j connected. All data cleared. Nodes: {count}")
print(f"OpenAI key: {os.environ.get('OPENAI_API_KEY', 'NOT SET')[:15]}...")

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv(Path(".").resolve().parent / ".env")
load_dotenv(Path(".").resolve() / ".env")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "workshop2024")

def ask_llm(prompt, system="You are a helpful assistant."):
    return llm.invoke([SystemMessage(content=system), HumanMessage(content=prompt)]).content

print("Setup complete!")

---
## Step 1: The Document

A company profile with people, roles, teams, skills, hobbies, projects, and partnerships.

In [ ]:
document = """
NovaMind Technologies — Company Profile

NovaMind Technologies is an AI startup founded in 2022, headquartered in Austin, Texas. The company
specializes in building intelligent document processing systems using large language models. NovaMind
has raised $45 million in Series B funding led by Sequoia Capital, with participation from Andreessen
Horowitz and Y Combinator.

Leadership Team:

Sarah Chen is the CEO and co-founder of NovaMind. She previously worked as VP of Engineering at
Databricks for 5 years and before that was a senior researcher at Google Brain. Sarah holds a PhD in
Computer Science from Stanford University. Outside of work, she is an avid rock climber and plays
competitive chess. She sits on the board of advisors for the AI Safety Institute.

Marcus Rodriguez is the CTO and co-founder. He was previously a principal engineer at Meta AI, where
he led the team that built the Llama fine-tuning infrastructure. Marcus studied at MIT and is known
in the developer community for his open-source contributions to the PyTorch ecosystem. He is a
marathon runner and mentors students at Code2040, a nonprofit that helps Black and Latinx students
break into tech.

Priya Sharma serves as the Head of Research. She joined from DeepMind, where she spent 4 years
working on retrieval-augmented generation systems. Priya completed her PhD at University of Oxford
and has published over 30 papers on knowledge representation. She enjoys painting watercolors and
is a volunteer at her local animal shelter.

James Park is the VP of Sales. Before NovaMind, he spent 8 years at Salesforce, most recently as
Regional Director for the West Coast. James graduated from Wharton School of Business and is a
licensed private pilot. He coaches youth basketball on weekends.

Lisa Nakamura is the Head of Product. She previously led product teams at Notion and before that at
Stripe. Lisa studied Human-Computer Interaction at Carnegie Mellon University. She is a food blogger
who runs the popular blog "TechFoodie" and practices yoga daily.

Engineering Teams:

The ML Platform team is led by Dr. Wei Zhang, who joined from NVIDIA's NeMo team. The team works on
the core inference engine that powers all NovaMind products. Wei is a contributor to the Hugging Face
Transformers library.

The Data Engineering team is managed by Carlos Mendez, who previously built data pipelines at
Snowflake. Carlos is also a competitive salsa dancer and DJ.

The Frontend team is led by Aisha Johnson, formerly of Figma. She specializes in building
collaborative real-time interfaces. Aisha is a published sci-fi author.

Key Products:

DocuMind is NovaMind's flagship product — an AI-powered document understanding platform used by
over 200 enterprise customers including Goldman Sachs, McKinsey, and Deloitte. It uses a proprietary
RAG pipeline combined with fine-tuned LLMs for document extraction, summarization, and Q&A.

GraphInsight is a newer product launched in 2024 that builds knowledge graphs from enterprise
documents automatically. It was developed in partnership with Neo4j and uses their graph database
as the backend. Priya Sharma led the research behind GraphInsight.

Partnerships and Ecosystem:

NovaMind has strategic partnerships with Microsoft Azure (cloud infrastructure), Neo4j (graph
database), and Anthropic (Claude API integration). The company is a member of the Partnership
on AI and contributes to the MLCommons benchmarking initiative.

Sarah Chen and Dario Amodei (CEO of Anthropic) have co-authored a white paper on responsible
AI deployment in enterprise settings. Marcus Rodriguez regularly collaborates with Harrison Chase
(founder of LangChain) on open-source agent frameworks.

Company Culture:

NovaMind has 120 employees across offices in Austin (HQ), San Francisco, and London. The company
hosts a monthly "Demo Day" where any employee can present a project. The engineering team uses
Python, TypeScript, and Rust as primary languages. The company offers a $5,000 annual learning
stipend and 4 weeks of PTO.
"""

print(f"Document: {len(document)} characters")
print(document[:300] + "\n...")

---
## Step 2: Define the Knowledge Graph Schema

Richer entity types — people, hobbies, skills, universities, not just tech entities.

In [ ]:
class Entity(BaseModel):
    name: str = Field(description="Entity name")
    type: str = Field(description="One of: PERSON, COMPANY, PRODUCT, UNIVERSITY, INVESTOR, CITY, SKILL, HOBBY, TEAM, NONPROFIT, ROLE")
    description: str = Field(description="One-line description")

class Relationship(BaseModel):
    source: str = Field(description="Source entity name")
    target: str = Field(description="Target entity name")
    type: str = Field(description="e.g. WORKS_AT, FOUNDED, STUDIED_AT, HAS_HOBBY, LEADS_TEAM, PREVIOUSLY_AT, INVESTED_IN, PARTNERS_WITH, COLLABORATES_WITH")
    description: str = Field(description="One-line description")

class KnowledgeGraph(BaseModel):
    entities: list[Entity] = Field(default_factory=list)
    relationships: list[Relationship] = Field(default_factory=list)

print("Schema ready!")
print("Entity types: PERSON, COMPANY, PRODUCT, UNIVERSITY, INVESTOR, CITY, SKILL, HOBBY, TEAM, NONPROFIT, ROLE")

---
## Step 3: Extract Knowledge Graph

In [ ]:
structured_llm = llm.with_structured_output(KnowledgeGraph)

print("Extracting entities and relationships... (takes ~15 seconds)\n")

kg = structured_llm.invoke([
    SystemMessage(content="""You are a knowledge graph extraction expert.
Extract ALL entities and relationships from the text.
Entity types: PERSON, COMPANY, PRODUCT, UNIVERSITY, INVESTOR, CITY, SKILL, HOBBY, TEAM, NONPROFIT, ROLE
Relationship types: WORKS_AT, CEO_OF, CTO_OF, FOUNDED, CO_FOUNDED, STUDIED_AT, HAS_HOBBY, LEADS_TEAM,
PREVIOUSLY_AT, INVESTED_IN, PARTNERS_WITH, COLLABORATES_WITH, BUILT, USES, LOCATED_IN, MEMBER_OF,
MENTORS_AT, ADVISES, CO_AUTHORED_WITH, HAS_SKILL, CUSTOMER_OF

Be thorough — extract people's hobbies, universities, previous companies, everything."""),
    HumanMessage(content=f"Extract the complete knowledge graph:\n\n{document}"),
])

print(f"Found {len(kg.entities)} entities and {len(kg.relationships)} relationships!")

### View Entities

In [ ]:
# Group by type
from collections import defaultdict
by_type = defaultdict(list)
for e in kg.entities:
    by_type[e.type].append(e.name)

for t, names in sorted(by_type.items()):
    print(f"\n{t} ({len(names)}):")
    for n in names:
        print(f"  - {n}")

### View Relationships

In [ ]:
for r in kg.relationships:
    print(f"  {r.source:<25} --[{r.type:<20}]--> {r.target}")

---
## Step 4: Load into Neo4j

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)

with driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
    print("Cleared old data\n")
    
    for e in kg.entities:
        s.run("CREATE (n:Entity {name: $name, type: $type, description: $desc})",
              name=e.name, type=e.type, desc=e.description)
    
    created = 0
    for r in kg.relationships:
        result = s.run(
            """MATCH (a:Entity {name: $src}), (b:Entity {name: $tgt})
               CREATE (a)-[:RELATES_TO {type: $rel, description: $desc}]->(b)
               RETURN count(*) AS c""",
            src=r.source, tgt=r.target, rel=r.type, desc=r.description)
        if result.single()["c"] > 0:
            created += 1
    
    nodes = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    edges = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]

print(f"Loaded: {nodes} nodes, {edges} relationships")
print(f"\nOpen http://localhost:7474 and run:")
print(f"  MATCH (n)-[r]->(m) RETURN n, r, m")

driver.close()

---
## Step 5: Visualize

In [ ]:
from pyvis.network import Network
import os

COLORS = {
    "PERSON": "#FF6B6B", "COMPANY": "#4ECDC4", "PRODUCT": "#45B7D1",
    "UNIVERSITY": "#96CEB4", "INVESTOR": "#F39C12", "CITY": "#FFEAA7",
    "SKILL": "#DDA0DD", "HOBBY": "#FF69B4", "TEAM": "#87CEEB",
    "NONPROFIT": "#98FB98", "ROLE": "#D3D3D3",
}

SHAPES = {
    "PERSON": "dot", "COMPANY": "diamond", "PRODUCT": "star",
    "UNIVERSITY": "triangle", "INVESTOR": "square", "CITY": "triangleDown",
    "HOBBY": "dot", "TEAM": "dot", "NONPROFIT": "dot",
}

net = Network(height="800px", width="100%", directed=True, bgcolor="#1a1a2e",
              font_color="white", cdn_resources="remote")
net.set_options("""
{
  "physics": {
    "barnesHut": { "gravitationalConstant": -5000, "springLength": 200, "springConstant": 0.04 },
    "stabilization": { "iterations": 100 }
  },
  "edges": {
    "color": { "color": "#636e72", "highlight": "#e17055" },
    "arrows": { "to": { "enabled": true, "scaleFactor": 0.8 } },
    "font": { "size": 10, "color": "#b2bec3", "strokeWidth": 0 },
    "smooth": { "type": "curvedCW", "roundness": 0.2 }
  },
  "nodes": {
    "font": { "size": 14, "color": "white", "face": "arial" },
    "borderWidth": 2, "borderWidthSelected": 4
  },
  "interaction": { "hover": true, "tooltipDelay": 100 }
}
""")

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    # Get degree for sizing
    degrees = {}
    for r in s.run("MATCH (n:Entity) OPTIONAL MATCH (n)-[r]-() RETURN n.name AS name, count(r) AS deg"):
        degrees[r["name"]] = r["deg"]
    
    for r in s.run("MATCH (n:Entity) RETURN n.name AS name, n.type AS type, n.description AS desc"):
        color = COLORS.get(r["type"], "#DFE6E9")
        shape = SHAPES.get(r["type"], "dot")
        size = 15 + degrees.get(r["name"], 0) * 5  # bigger = more connected
        tooltip = f"<b>{r['name']}</b><br>Type: {r['type']}<br>{r['desc']}"
        net.add_node(r["name"], label=r["name"], color=color, shape=shape,
                     size=size, title=tooltip)
    for r in s.run("MATCH (a)-[r]->(b) RETURN a.name AS src, b.name AS tgt, r.type AS type, r.description AS desc"):
        net.add_edge(r["src"], r["tgt"], label=r["type"],
                     title=f"{r['type']}: {r['desc']}")
driver.close()

# Add legend
for etype, color in COLORS.items():
    net.add_node(f"_legend_{etype}", label=etype, color=color,
                 shape=SHAPES.get(etype, "dot"), size=8, x=-800, y=-400 + list(COLORS.keys()).index(etype) * 40,
                 physics=False, font={"size": 10})

html_path = os.path.abspath("company_kg.html")
net.save_graph(html_path)
print(f"Saved to: {html_path}")
print(f"Open in browser: {html_path}")

def graph_rag(question):
    """Graph RAG: extract entities from question → fetch their neighborhood → LLM answers."""
    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    
    # Step 1: Ask LLM to identify key entities in the question
    entity_str = ask_llm(
        f"List the key entity names from this question, one per line:\n{question}",
        system="Return ONLY entity names, one per line. No explanations. No bullet points."
    )
    search_terms = [e.strip().strip("-•* ") for e in entity_str.strip().split("\n") if e.strip()]
    print(f"Entities from question: {search_terms}")
    
    # Step 2: Fetch ONLY the neighborhood of those entities (not the full graph)
    triples = []
    with driver.session() as s:
        for term in search_terms:
            for r in s.run(
                """MATCH (a:Entity)-[r:RELATES_TO]->(b:Entity)
                   WHERE toLower(a.name) CONTAINS toLower($t)
                      OR toLower(b.name) CONTAINS toLower($t)
                   RETURN a.name AS source, r.type AS rel, b.name AS target""",
                t=term
            ):
                triples.append(f"{r['source']} --[{r['rel']}]--> {r['target']}")
    driver.close()
    
    # Deduplicate
    triples = list(set(triples))
    print(f"Fetched {len(triples)} relevant triples (not the full graph)\n")
    for t in triples:
        print(f"  {t}")
    
    # Step 3: LLM answers from the relevant subgraph
    context = "\n".join(triples)
    answer = ask_llm(
        f"Knowledge Graph (relevant subgraph):\n{context}\n\nQuestion: {question}",
        system="Answer using ONLY the graph relationships provided. Trace paths between entities."
    )
    return answer

print("graph_rag() ready — fetches only relevant subgraph, not full graph!")

In [ ]:
### Q1: Sarah Chen's background

print(graph_rag("Where did Sarah Chen work before, where did she study, and what are her hobbies?"))

In [ ]:
### Q2: Marcus Rodriguez's connections

print(graph_rag("Who does Marcus Rodriguez collaborate with, where did he study, and what does he mentor?"))

In [ ]:
### Q3: Priya Sharma and Lisa Nakamura — compare their backgrounds

print(graph_rag("Compare Priya Sharma and Lisa Nakamura — where did each study, work before, and what are their hobbies?"))

In [ ]:
### Q4: NovaMind's ecosystem — partners and products

print(graph_rag("What are NovaMind's products, who are their partners like Anthropic and Neo4j, and who invested in them?"))

In [ ]:
### Q5: Wei Zhang and Carlos Mendez — where did they come from?

print(graph_rag("Where did Wei Zhang and Carlos Mendez previously work? What do they do at NovaMind?"))

In [ ]:
### Q6: Multi-hop — What connects Sarah Chen to Anthropic?

print(graph_rag("What connects Sarah Chen to Anthropic? Trace the path through Dario Amodei and NovaMind."))

In [ ]:
print(graph_rag("What connects Sarah Chen to Anthropic? Trace the path."))

---
## Key Takeaway

From a single text document, we:
1. Extracted **people, companies, products, hobbies, universities, skills** automatically
2. Mapped **relationships**: who works where, studied where, has what hobbies, collaborates with whom
3. Loaded into Neo4j and queried with **Graph RAG**

Multi-hop questions like *"What connects Sarah Chen to Anthropic?"* require graph traversal:
`Sarah Chen -> co-authored with -> Dario Amodei -> CEO of -> Anthropic`

**No vector search can do this. Only a Knowledge Graph can.**